# Fine-tune Qwen2.5-VL-3B-Instruct cho OCR chu viet tay tieng Viet (ms-swift, Colab A100)

- Dataset: `tranhuy67896262/Viet-Handwriting-OCR-v2-local` (columns: `image` + `text`, ~60k train)
- Base: `Qwen/Qwen2.5-VL-3B-Instruct`
- Framework: **ms-swift** (`swift sft` LoRA bf16 -> `swift infer` -> `swift export --merge_lora` -> push Hub)
- Output: `tranhuy67896262/qwen25vl-3b-vi-hwr-lora-v1`

**Chay:** Upload notebook nay len Colab, Runtime = A100 + High-RAM, them Secret `HF_TOKEN` (khoa ben trai) -> Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no-gpu')"

In [ ]:
# ===== 1. CONFIG (da fix cung cho dataset cua ban) =====
HF_DATASET_ID = "tranhuy67896262/Viet-Handwriting-OCR-v2-local"
MODEL_ID = "tranhuy67896262/Qwen2.5-VL-3B-Instruct-private"
HF_OUTPUT_REPO = "tranhuy67896262/qwen25vl-3b-vi-hwr-lora-v1"

OUTPUT_DIR = "/content/output"
DATA_DIR = "/content/data"
IMG_DIR = f"{DATA_DIR}/images"
TRAIN_JSONL = f"{DATA_DIR}/train_swift.jsonl"
VAL_JSONL = f"{DATA_DIR}/val_swift.jsonl"

OCR_PROMPT = "Doc va ghi lai chinh xac toan bo chu viet tay tieng Viet trong anh. Chi tra ve noi dung van ban, khong giai thich them."

LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = "1e-4"
NUM_EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_LENGTH = 1024
MAX_TRAIN_SAMPLES = 0
TRAIN_OFFSET = 0  # chunked training: 0, 5000, 10000...
MAX_VAL_SAMPLES = 1000
print("config ok")

In [ ]:
# ===== 2. INSTALL ms-swift (Colab, ~3-5 phut) =====
!pip install -q ms-swift qwen-vl-utils datasets pyarrow pillow huggingface_hub
!python -c 'import swift; print(swift.__version__)'

In [ ]:
# ===== 3. HF LOGIN qua Colab Secrets (khong hardcode token) =====
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login ok")

In [ ]:
# ===== 4. CONVERT HF image+text -> swift JSONL (messages + images) =====
# Schema: {image: {bytes, path}, text: str}, 7 shard train (~8.4k/shard)
import json, os
from datasets import load_dataset

os.makedirs(IMG_DIR, exist_ok=True)

train_ds = load_dataset(HF_DATASET_ID, split="train")
val_ds = load_dataset(HF_DATASET_ID, split="test")
print(f"train={len(train_ds)} val={len(val_ds)}")
print("sample:", repr(str(train_ds[0]['text'])[:150]))

if TRAIN_OFFSET or MAX_TRAIN_SAMPLES:
    end = len(train_ds) if not MAX_TRAIN_SAMPLES else TRAIN_OFFSET + MAX_TRAIN_SAMPLES
    train_ds = train_ds.select(range(TRAIN_OFFSET, min(end, len(train_ds))))
if MAX_VAL_SAMPLES and len(val_ds) > MAX_VAL_SAMPLES:
    val_ds = val_ds.select(range(MAX_VAL_SAMPLES))

def dump_swift(ds, out_path, img_prefix, start_idx=0):
    n_skip = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(ds):
            text = (row.get("text") or "").strip()
            img = row.get("image")
            if not text or img is None:
                n_skip += 1
                continue
            img_path = os.path.join(IMG_DIR, f"{img_prefix}_{start_idx + i}.jpg")
            if not os.path.exists(img_path):
                try:
                    img.convert("RGB").save(img_path, "JPEG")
                except Exception:
                    n_skip += 1
                    continue
            rec = {"messages": [{"role": "user", "content": "<image>" + OCR_PROMPT}, {"role": "assistant", "content": text}], "images": [img_path]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"wrote {out_path} (skip={n_skip})")

dump_swift(train_ds, TRAIN_JSONL, "train", start_idx=TRAIN_OFFSET)
dump_swift(val_ds, VAL_JSONL, "val")
!wc -l /content/data/*.jsonl && head -c 400 /content/data/train_swift.jsonl

In [ ]:
# ===== 5. TRAIN bang ms-swift (LoRA bf16, single A100) =====
!swift sft \
    --model {MODEL_ID} \
    --template qwen2_5_vl \
    --dataset {TRAIN_JSONL} \
    --val_dataset {VAL_JSONL} \
    --use_hf true \
    --push_to_hub true \
    --hub_model_id {HF_OUTPUT_REPO} \
    --hub_strategy checkpoint \
    --train_type lora \
    --lora_rank {LORA_R} \
    --lora_alpha {LORA_ALPHA} \
    --target_modules all-linear \
    --torch_dtype bfloat16 \
    --num_train_epochs {NUM_EPOCHS} \
    --per_device_train_batch_size {BATCH_SIZE} \
    --per_device_eval_batch_size {BATCH_SIZE} \
    --gradient_accumulation_steps {GRAD_ACCUM} \
    --learning_rate {LEARNING_RATE} \
    --warmup_ratio 0.05 \
    --max_length {MAX_LENGTH} \
    --max_pixels 786432 \
    --eval_steps 50 \
    --save_steps 50 \
    --save_total_limit 3 \
    --logging_steps 10 \
    --output_dir {OUTPUT_DIR} \
    --dataloader_num_workers 2 \
    --gradient_checkpointing true \
    --attn_impl flash_attn
# Neu OOM: giam BATCH_SIZE 4->2, hoac MAX_LENGTH 1024->768

In [ ]:
# ===== 6. TEST checkpoint LoRA tren 1 anh val (swift infer batch) =====
import glob, json
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*/adapter_config.json"))
print(f"lora checkpoints: {len(ckpts)}")
LAST_CKPT = ckpts[-1].replace("/adapter_config.json", "") if ckpts else OUTPUT_DIR
print("LAST_CKPT =", LAST_CKPT)

row = json.loads(open(VAL_JSONL, encoding="utf-8").readline())
TEST_IMG = row["images"][0]
print("img:", TEST_IMG)
print("ref:", row["messages"][1]["content"][:150])
with open("/content/data/test_one.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps({"messages": [{"role": "user", "content": "<image>" + OCR_PROMPT}], "images": [TEST_IMG]}, ensure_ascii=False) + "\n")
print("wrote test_one.jsonl")

In [ ]:
!swift infer --model {MODEL_ID} --template qwen2_5_vl --adapters {LAST_CKPT} --use_hf true --val_dataset /content/data/test_one.jsonl --torch_dtype bfloat16 --max_new_tokens 256

In [ ]:
# ===== 7. MERGE LoRA + PUSH len HF Hub =====
!swift export --adapters {LAST_CKPT} --merge_lora true --output_dir /content/merged
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_OUTPUT_REPO, exist_ok=True, private=False)
api.upload_folder(folder_path="/content/merged", repo_id=HF_OUTPUT_REPO, commit_message="qwen2.5-vl-3b vi-handwriting-ocr lora merged (ms-swift)")
print(f"Merged -> https://huggingface.co/{HF_OUTPUT_REPO}")
api.upload_folder(folder_path=LAST_CKPT, repo_id=HF_OUTPUT_REPO, commit_message="lora adapter", path_in_repo="lora")
print("Adapter -> cung repo, thu muc /lora")